# Lab | Web Scraping

Welcome to the "Books to Scrape" Web Scraping Adventure Lab!

**Objective**

In this lab, we will embark on a mission to unearth valuable insights from the data available on Books to Scrape, an online platform showcasing a wide variety of books. As data analyst, you have been tasked with scraping a specific subset of book data from Books to Scrape to assist publishing companies in understanding the landscape of highly-rated books across different genres. Your insights will help shape future book marketing strategies and publishing decisions.

**Background**

In a world where data has become the new currency, businesses are leveraging big data to make informed decisions that drive success and profitability. The publishing industry, much like others, utilizes data analytics to understand market trends, reader preferences, and the performance of books based on factors such as genre, author, and ratings. Books to Scrape serves as a rich source of such data, offering detailed information about a diverse range of books, making it an ideal platform for extracting insights to aid in informed decision-making within the literary world.

**Task**

Your task is to create a Python script using BeautifulSoup and pandas to scrape Books to Scrape book data, focusing on book ratings and genres. The script should be able to filter books with ratings above a certain threshold and in specific genres. Additionally, the script should structure the scraped data in a tabular format using pandas for further analysis.

**Expected Outcome**

A function named `scrape_books` that takes two parameters: `min_rating` and `max_price`. The function should scrape book data from the "Books to Scrape" website and return a `pandas` DataFrame with the following columns:

**Expected Outcome**

- A function named `scrape_books` that takes two parameters: `min_rating` and `max_price`.
- The function should return a DataFrame with the following columns:
  - **UPC**: The Universal Product Code (UPC) of the book.
  - **Title**: The title of the book.
  - **Price (£)**: The price of the book in pounds.
  - **Rating**: The rating of the book (1-5 stars).
  - **Genre**: The genre of the book.
  - **Availability**: Whether the book is in stock or not.
  - **Description**: A brief description or product description of the book (if available).
  
You will execute this script to scrape data for books with a minimum rating of `4.0 and above` and a maximum price of `£20`. 

Remember to experiment with different ratings and prices to ensure your code is versatile and can handle various searches effectively!

**Resources**

- [Beautiful Soup Documentation](https://www.crummy.com/software/BeautifulSoup/bs4/doc/)
- [Pandas Documentation](https://pandas.pydata.org/pandas-docs/stable/index.html)
- [Books to Scrape](https://books.toscrape.com/)


**Hint**

Your first mission is to familiarize yourself with the **Books to Scrape** website. Navigate to [Books to Scrape](http://books.toscrape.com/) and explore the available books to understand their layout and structure. 

Next, think about how you can set parameters for your data extraction:

- **Minimum Rating**: Focus on books with a rating of 4.0 and above.
- **Maximum Price**: Filter for books priced up to £20.

After reviewing the site, you can construct a plan for scraping relevant data. Pay attention to the details displayed for each book, including the title, price, rating, and availability. This will help you identify the correct HTML elements to target with your scraping script.

Make sure to build your scraping URL and logic based on the patterns you observe in the HTML structure of the book listings!


---

**Best of luck! Immerse yourself in the world of books, and may the data be with you!**

**Important Note**:

In the fast-changing online world, websites often update and change their structures. When you try this lab, the **Books to Scrape** website might differ from what you expect.

If you encounter issues due to these changes, like new rules or obstacles preventing data extraction, don’t worry! Get creative.

You can choose another website that interests you and is suitable for scraping data. Options like Wikipedia, The New York Times, or even library databases are great alternatives. The main goal remains the same: extract useful data and enhance your web scraping skills while exploring a source of information you enjoy. This is your opportunity to practice and adapt to different web environments!

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# url del catalogo (las paginas van de page-1 a page-50)
url = "https://books.toscrape.com/catalogue/"

# pruebo con la primera pagina
r = requests.get(url + "page-1.html")
print(r.status_code)  # tiene que salir 200

# uso content porque con text el simbolo £ sale raro
soup = BeautifulSoup(r.content, "html.parser")

# cada libro esta en un article con clase product_pod
libros = soup.find_all("article", class_="product_pod")
print(len(libros))

200
20


In [2]:
# las estrellas vienen en ingles dentro de la clase, las paso a numero
estrellas = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

for libro in libros:
    titulo = libro.h3.a["title"]  # el titulo entero esta en el atributo title
    precio = float(libro.find("p", class_="price_color").text.replace("£", ""))
    rating = estrellas[libro.find("p", class_="star-rating")["class"][1]]
    print(titulo, precio, rating)

A Light in the Attic 51.77 3
Tipping the Velvet 53.74 1
Soumission 50.1 1
Sharp Objects 47.82 4
Sapiens: A Brief History of Humankind 54.23 5
The Requiem Red 22.65 1
The Dirty Little Secrets of Getting Your Dream Job 33.34 4
The Coming Woman: A Novel Based on the Life of the Infamous Feminist, Victoria Woodhull 17.93 3
The Boys in the Boat: Nine Americans and Their Epic Quest for Gold at the 1936 Berlin Olympics 22.6 4
The Black Maria 52.15 1
Starving Hearts (Triangular Trade Trilogy, #1) 13.99 2
Shakespeare's Sonnets 20.66 4
Set Me Free 17.46 5
Scott Pilgrim's Precious Little Life (Scott Pilgrim #1) 52.29 5
Rip it Up and Start Again 35.02 5
Our Band Could Be Your Life: Scenes from the American Indie Underground, 1981-1991 57.25 3
Olio 23.88 1
Mesaerion: The Best Science Fiction Stories 1800-1849 37.59 1
Libertarianism for Beginners 51.33 2
It's Only the Himalayas 45.17 2


In [3]:
# pruebo el filtro con 4 estrellas o mas y 20 libras como maximo
min_rating = 4
max_price = 20

enlaces = []
for libro in libros:
    precio = float(libro.find("p", class_="price_color").text.replace("£", ""))
    rating = estrellas[libro.find("p", class_="star-rating")["class"][1]]
    if rating >= min_rating and precio <= max_price:
        enlaces.append(url + libro.h3.a["href"])  # el href es relativo, le pego la url delante

print(len(enlaces))
print(enlaces)

1
['https://books.toscrape.com/catalogue/set-me-free_988/index.html']


In [4]:
# abro la ficha del primer libro que paso el filtro
r2 = requests.get(enlaces[0])
ficha = BeautifulSoup(r2.content, "html.parser")

titulo = ficha.find("h1").text
upc = ficha.find("table").find("td").text  # la primera fila de la tabla es el UPC
genero = ficha.find("ul", class_="breadcrumb").find_all("a")[2].text  # home > books > genero
stock = ficha.find("p", class_="instock availability").text.strip()

# algunos libros no tienen descripcion, por eso miro primero si existe
desc = ficha.find("div", id="product_description")
if desc:
    descripcion = desc.find_next_sibling("p").text
else:
    descripcion = ""

print(titulo)
print(upc)
print(genero)
print(stock)
print(descripcion[:100])  # solo el principio, que es muy larga

Set Me Free
ce6396b0f23f6ecc
Young Adult
In stock (19 available)
Aaron Ledbetter’s future had been planned out for him since before he was born. Each year, the Ledbe


In [5]:
def scrape_books(min_rating, max_price):
    url = "https://books.toscrape.com/catalogue/"
    estrellas = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
    datos = []

    # hay 50 paginas
    for pagina in range(1, 51):
        r = requests.get(url + f"page-{pagina}.html")
        soup = BeautifulSoup(r.content, "html.parser")
        libros = soup.find_all("article", class_="product_pod")

        for libro in libros:
            precio = float(libro.find("p", class_="price_color").text.replace("£", ""))
            rating = estrellas[libro.find("p", class_="star-rating")["class"][1]]

            # si no pasa el filtro paso al siguiente libro
            if rating < min_rating or precio > max_price:
                continue

            # entro en la ficha para sacar lo que falta
            r2 = requests.get(url + libro.h3.a["href"])
            ficha = BeautifulSoup(r2.content, "html.parser")

            desc = ficha.find("div", id="product_description")
            if desc:
                descripcion = desc.find_next_sibling("p").text
            else:
                descripcion = ""

            datos.append({
                "UPC": ficha.find("table").find("td").text,
                "Title": ficha.find("h1").text,
                "Price (£)": precio,
                "Rating": rating,
                "Genre": ficha.find("ul", class_="breadcrumb").find_all("a")[2].text,
                "Availability": ficha.find("p", class_="instock availability").text.strip(),
                "Description": descripcion
            })

        print("pagina", pagina, "hecha")  # para ver que avanza

    return pd.DataFrame(datos)

In [6]:
# lo que pide el lab: 4 estrellas o mas y maximo 20 libras
df = scrape_books(4, 20)
print(df.shape)
df.head()

pagina 1 hecha
pagina 2 hecha
pagina 3 hecha
pagina 4 hecha
pagina 5 hecha
pagina 6 hecha
pagina 7 hecha
pagina 8 hecha
pagina 9 hecha
pagina 10 hecha
pagina 11 hecha
pagina 12 hecha
pagina 13 hecha
pagina 14 hecha
pagina 15 hecha
pagina 16 hecha
pagina 17 hecha
pagina 18 hecha
pagina 19 hecha
pagina 20 hecha
pagina 21 hecha
pagina 22 hecha
pagina 23 hecha
pagina 24 hecha
pagina 25 hecha
pagina 26 hecha
pagina 27 hecha
pagina 28 hecha
pagina 29 hecha
pagina 30 hecha
pagina 31 hecha
pagina 32 hecha
pagina 33 hecha
pagina 34 hecha
pagina 35 hecha
pagina 36 hecha
pagina 37 hecha
pagina 38 hecha
pagina 39 hecha
pagina 40 hecha
pagina 41 hecha
pagina 42 hecha
pagina 43 hecha
pagina 44 hecha
pagina 45 hecha
pagina 46 hecha
pagina 47 hecha
pagina 48 hecha
pagina 49 hecha
pagina 50 hecha
(75, 7)


,UPC,Title,Price (£),Rating,Genre,Availability,Description
0,ce6396b0f23f6ecc,Set Me Free,17.46,5,Young Adult,In stock (19 available),Aaron Ledbetter’s future had been planned out ...
1,6258a1f6a6dcfe50,The Four Agreements: A Practical Guide to Pers...,17.66,5,Spirituality,In stock (18 available),"In The Four Agreements, don Miguel Ruiz reveal..."
2,6be3beb0793a53e7,Sophie's World,15.94,5,Philosophy,In stock (18 available),A page-turning novel that is also an explorati...
3,657fe5ead67a7767,Untitled Collection: Sabbath Poems 2014,14.27,4,Poetry,In stock (16 available),"More than thirty-five years ago, when the weat..."
4,51653ef291ab7ddc,This One Summer,19.49,4,Sequential Art,In stock (16 available),"Every summer, Rose goes with her mom and dad t..."


In [7]:
# que generos salen mas
print(df["Genre"].value_counts().head())

# precio medio y cuantos de 4 y de 5 estrellas
print(df["Price (£)"].mean())
print(df["Rating"].value_counts())

Genre
Default           10
Sequential Art     9
Young Adult        7
Nonfiction         6
Fiction            5
Name: count, dtype: int64
14.570933333333336
Rating
5    42
4    33
Name: count, dtype: int64


In [8]:
# pruebo con otros valores para ver que la funcion vale para cualquier busqueda
df2 = scrape_books(5, 15)
print(df2.shape)
df2.head()

pagina 1 hecha
pagina 2 hecha
pagina 3 hecha
pagina 4 hecha
pagina 5 hecha
pagina 6 hecha
pagina 7 hecha
pagina 8 hecha
pagina 9 hecha
pagina 10 hecha
pagina 11 hecha
pagina 12 hecha
pagina 13 hecha
pagina 14 hecha
pagina 15 hecha
pagina 16 hecha
pagina 17 hecha
pagina 18 hecha
pagina 19 hecha
pagina 20 hecha
pagina 21 hecha
pagina 22 hecha
pagina 23 hecha
pagina 24 hecha
pagina 25 hecha
pagina 26 hecha
pagina 27 hecha
pagina 28 hecha
pagina 29 hecha
pagina 30 hecha
pagina 31 hecha
pagina 32 hecha
pagina 33 hecha
pagina 34 hecha
pagina 35 hecha
pagina 36 hecha
pagina 37 hecha
pagina 38 hecha
pagina 39 hecha
pagina 40 hecha
pagina 41 hecha
pagina 42 hecha
pagina 43 hecha
pagina 44 hecha
pagina 45 hecha
pagina 46 hecha
pagina 47 hecha
pagina 48 hecha
pagina 49 hecha
pagina 50 hecha
(23, 7)


,UPC,Title,Price (£),Rating,Genre,Availability,Description
0,0fa6dceead7ce47a,"Princess Jellyfish 2-in-1 Omnibus, Vol. 01 (Pr...",13.61,5,Sequential Art,In stock (16 available),THE LONG-AWAITED STORY OF FANGIRLS TAKING ON T...
1,0e691eda369f4e09,Princess Between Worlds (Wide-Awake Princess #5),13.34,5,Fantasy,In stock (16 available),Just as Annie and Liam are busy making plans t...
2,3bebf34ee9330cbd,The Third Wave: An Entrepreneur’s Vision of th...,12.61,5,Business,In stock (15 available),NEW YORK TIMES BESTSELLERWALL STREET JOURNAL B...
3,d195e7f4ed697350,Agnostic: A Spirited Manifesto,12.51,5,Nonfiction,In stock (15 available),A widely admired writer on religion celebrates...
4,6a31307a81e8f3a8,Walt Disney's Alice in Wonderland,12.96,5,Childrens,In stock (14 available),The fantastical tale of a young girl chasing h...
